# 03 Matching Comparison — сравнение способов определить тип пары

Эта тетрадка начинается после ручной разметки. На входе у нас есть CSV с парами товаров и твоими метками: тот же это базовый товар или другой товар.

Цель тетрадки простая: взять первые методы сравнения пар и посмотреть, как часто они угадывают бинарную разметку.

Здесь сравниваются два простых ориентира:

- `rule_based_fuzzy` — правила на похожести названий плюс бренд. Это не модель, а понятная нижняя планка.
- `bi_encoder_zero_shot` — сравнение текстов через готовую модель `e5-small`. Мы её не обучаем на наших данных, поэтому это тоже только начальный ориентир.

Главный вопрос этой тетрадки: достаточно ли таких простых подходов, или нужен более сильный следующий метод. Фасовка и multipack больше не являются отдельным ML-классом: они разбираются после модели deterministic правилами.


## Мини-словарь перед запуском

`label` — твоя ручная метка, то есть правильный ответ.

`predicted_label` — метка, которую поставил метод.

`exact_duplicate` — тот же базовый товар для модели, даже если фасовка или multipack отличаются.

`different_product` — разные товары.

`uncertain` — спорная пара. В обычные метрики она не входит, чтобы не заставлять метод угадывать то, в чём мы сами не уверены.

`precision` — насколько можно верить предсказаниям класса. Например, из всех пар, которые метод назвал `exact_duplicate`, какая доля действительно была `exact_duplicate`.

`recall` — насколько хорошо метод находит все пары класса. Например, из всех настоящих `exact_duplicate`, сколько метод смог найти.

`F1` — одно число, которое объединяет `precision` и `recall`. Удобно для сравнения, но его нельзя читать без таблицы ошибок.

`false merge` — опасная ошибка: настоящие разные товары метод предлагает связать как один товар.


## Как устроена проверка

Мы делим размеченные пары на две части:

- `dev` — часть для подбора порога. На ней метод учится, насколько строгим быть.
- `test` — отложенная часть для честной проверки. На ней мы смотрим итоговые цифры.

Это нужно, чтобы не подгонять порог под все данные сразу. Если подобрать порог и отчитаться на одних и тех же строках, метрика будет слишком оптимистичной.


## Блок кода 1. Подготовка окружения

Эта ячейка подключает библиотеки, находит корень проекта и импортирует нужные функции из `research/dedup`.

Если здесь ошибка, чаще всего причина простая: тетрадка запущена не из папки проекта или не установлен пакет для ноутбуков.


In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import math
import os
import sys
import time
from typing import Any

from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    CROSS_ENCODER_BACKEND,
    SENTENCE_TRANSFORMER_BACKEND,
    TRANSFORMERS_AUTO_MODEL_BACKEND,
    BiEncoderMatcher,
    FusionConfig,
    ModelManager,
    POLZA_EMBEDDING_BACKEND,
    RuleBasedMatcher,
    classification_report_df,
    confusion_matrix_df,
    decide_label,
)
from research.dedup.matchers.bi_encoder import BiEncoderConfig

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)


## Блок кода 2. Настройки тетрадки

Здесь задаются пути к файлам и основные параметры проверки.

Самое важное:

- `LABELING_PATH` — файл с твоей разметкой.
- `RUN_BI_ENCODER` — запускать ли модель для сравнения текстов.
- `BI_ENCODER_MODEL` — alias из `research.dedup.model_registry` или прямой model id.
- `DEDUP_BI_ENCODER_BACKEND=polza_embedding` — если передаёшь прямой Polza model id вместо alias.
- `POLZA_API_KEY` или `POLZA_AI_API_KEY` — ключ для online-моделей Polza.ai.
- `POLZA_BASE_URL` — override base URL, по умолчанию `https://polza.ai/api/v1`.
- `DEDUP_MODEL_CACHE_DIR` — куда локально складываются скачанные local-модели; по умолчанию `research/dedup/models/`.
- `DEDUP_MODEL_LOCAL_ONLY=1` — offline-режим: не скачивать модель, а брать только уже лежащую в кэше.
- `DEV_FRACTION` — какая часть размеченных пар идёт на подбор порога.
- `TARGET_EXACT_PRECISION` — желаемая строгость для точных дублей.

Если у тебя не установлен `sentence-transformers`, метод `bi_encoder_zero_shot` не сможет нормально отработать.


In [ ]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
LABELING_PATH = DATA_DIR / "labeling_sauces.csv"
PREDICTIONS_PATH = DATA_DIR / "matching_predictions_sauces.csv"
SUMMARY_PATH = DATA_DIR / "matching_summary_sauces.csv"
FALSE_MERGES_PATH = DATA_DIR / "matching_false_merges_sauces.csv"

EVAL_LABELS = ["exact_duplicate", "different_product"]
MERGE_LIKE_LABELS = {"exact_duplicate"}

MODEL_MANAGER = ModelManager()
RUN_BI_ENCODER = os.environ.get("DEDUP_RUN_BI_ENCODER", "1") == "1"
BI_ENCODER_MODEL = os.environ.get("DEDUP_BI_ENCODER_MODEL", "bi_encoder_e5_small")
BI_ENCODER_MODEL_BACKEND = os.environ.get("DEDUP_BI_ENCODER_BACKEND", "").strip() or None
BI_ENCODER_MODEL_SPEC = MODEL_MANAGER.resolve_embedding_model(BI_ENCODER_MODEL, backend=BI_ENCODER_MODEL_BACKEND)
RANDOM_STATE = int(os.environ.get("DEDUP_EVAL_RANDOM_STATE", "42"))
DEV_FRACTION = float(os.environ.get("DEDUP_EVAL_DEV_FRACTION", "0.60"))
TARGET_EXACT_PRECISION = float(os.environ.get("DEDUP_TARGET_EXACT_PRECISION", "0.85"))
THRESHOLD_GRID = [round(value / 100, 2) for value in range(60, 101)]

print(f"Labeling path: {LABELING_PATH}")
print(f"Run bi-encoder: {RUN_BI_ENCODER}")
print(f"Bi-encoder model alias/input: {BI_ENCODER_MODEL}")
print(f"Bi-encoder model id: {BI_ENCODER_MODEL_SPEC.model_name}")
print(f"Bi-encoder backend: {BI_ENCODER_MODEL_SPEC.backend}")
if BI_ENCODER_MODEL_SPEC.backend == POLZA_EMBEDDING_BACKEND:
    print(f"Polza base URL: {MODEL_MANAGER.polza_base_url}")
print(f"Model cache dir: {MODEL_MANAGER.cache_dir}")
print(f"Local-only model loading: {MODEL_MANAGER.local_files_only}")
print(f"Dev fraction: {DEV_FRACTION:.0%}; random_state={RANDOM_STATE}")
print(f"Target exact_duplicate precision for calibration: {TARGET_EXACT_PRECISION:.0%}")


## Блок кода 3. Загрузка и первичная проверка разметки

Эта ячейка читает `labeling_sauces.csv`, проверяет колонку `label`, убирает `uncertain` из расчёта метрик и делит строки на `dev` и `test`.

В выводе нужно смотреть:

- сколько всего строк в разметке;
- сколько строк реально попало в метрики;
- сколько строк отброшено как `uncertain`;
- как классы распределились между `dev` и `test`.

Если классов в `test` очень мало, итоговые цифры будут шумными.


In [ ]:
def load_labeled_pairs(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        status = pd.DataFrame([
            {
                "status": "missing_labeling_file",
                "message": f"Файл {path} пока не найден. Выполните 02_labeling_dataset.ipynb и заполните label.",
                "rows_total": 0,
                "rows_usable_for_metrics": 0,
                "rows_ignored_non_binary": 0,
            }
        ])
        return pd.DataFrame(), status

    frame = pd.read_csv(path)
    if "label" not in frame.columns:
        status = pd.DataFrame([
            {
                "status": "missing_label_column",
                "message": "В файле нет колонки label. Перегенерируйте labeling dataset из notebook-2.",
                "rows_total": len(frame),
                "rows_usable_for_metrics": 0,
                "rows_ignored_non_binary": 0,
            }
        ])
        return pd.DataFrame(), status

    labels = frame["label"].fillna("").astype(str).str.strip()
    labeled = frame[labels.isin(EVAL_LABELS)].copy()
    labeled["label"] = labels[labels.isin(EVAL_LABELS)].to_numpy()
    ignored_count = int(labels.ne("").sum() - len(labeled))
    status_name = "ready" if not labeled.empty else "empty_or_not_reviewed_yet"
    message = (
        "Gold-set готов для метрик."
        if not labeled.empty
        else "Binary labels пока не заполнены: метрики ниже будут заглушками, notebook не падает."
    )
    status = pd.DataFrame([
        {
            "status": status_name,
            "message": message,
            "rows_total": len(frame),
            "rows_usable_for_metrics": len(labeled),
            "rows_ignored_non_binary": ignored_count,
        }
    ])
    return labeled.reset_index(drop=True), status


def add_stratified_eval_split(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.assign(eval_split=pd.Series(dtype="string"))
    parts: list[pd.DataFrame] = []
    for _, group in frame.groupby("label", sort=False):
        shuffled = group.sample(frac=1.0, random_state=RANDOM_STATE)
        if len(shuffled) == 1:
            dev = shuffled.copy()
            dev["eval_split"] = "dev"
            parts.append(dev)
            continue
        dev_count = int(round(len(shuffled) * DEV_FRACTION))
        dev_count = min(max(1, dev_count), len(shuffled) - 1)
        dev = shuffled.iloc[:dev_count].copy()
        test = shuffled.iloc[dev_count:].copy()
        dev["eval_split"] = "dev"
        test["eval_split"] = "test"
        parts.extend([dev, test])
    return pd.concat(parts).sort_index().reset_index(drop=True)


labeled_pairs, labeling_status = load_labeled_pairs(LABELING_PATH)
labeled_pairs = add_stratified_eval_split(labeled_pairs)

display(labeling_status)
if labeled_pairs.empty:
    display(pd.DataFrame(columns=["label", "pairs"]))
else:
    display(labeled_pairs["label"].value_counts().rename_axis("label").reset_index(name="pairs"))
    display(pd.crosstab(labeled_pairs["eval_split"], labeled_pairs["label"]))


## Блок кода 4. Список методов, которые будем сравнивать

Эта ячейка создаёт методы сравнения пар и показывает, доступны ли они в текущем окружении.

Важно смотреть на строку `bi_encoder_zero_shot`:

- `available=True` означает, что пакет найден;
- `available=False` означает, что модельный способ будет пропущен.

Если метод пропущен, это не портит rule-based проверку, но сравнение с моделью будет неполным.


In [ ]:
matchers = [RuleBasedMatcher()]
if RUN_BI_ENCODER:
    matchers.append(BiEncoderMatcher(BiEncoderConfig(model_name=BI_ENCODER_MODEL, model_backend=BI_ENCODER_MODEL_BACKEND)))

method_status = []
for matcher in matchers:
    status = matcher.status()
    method_status.append({"method": matcher.name, "available": status.available, "status": status.message})

method_status_df = pd.DataFrame(method_status)
display(method_status_df)


## Блок кода 5. Вспомогательные функции для предсказаний

Эта ячейка не считает итоговые цифры. Она задаёт правила, как из численного сходства пары получить одну из трёх меток.

Логика такая:

1. Метод считает `score` — насколько пара похожа.
2. Дальше `fusion` добавляет простые признаки: бренд, вес, количество штук в наборе.
3. На выходе получается `predicted_label`.

Код здесь нужен, чтобы одинаково обработать оба метода: простой rule-based и модельный bi-encoder.


In [5]:
def _base_fusion_config(matcher: Any) -> FusionConfig:
    config = getattr(matcher, "config", None)
    return getattr(config, "fusion", FusionConfig())


def _fallback_label(matcher: Any) -> str:
    config = getattr(matcher, "config", None)
    return getattr(config, "uncertain_fallback_label", "different_product")


def _predict_from_score(matcher: Any, row: pd.Series, score: float, *, threshold_high: float | None = None) -> str:
    if math.isnan(score):
        return "different_product"
    fusion_config = _base_fusion_config(matcher)
    if threshold_high is not None:
        fusion_config = replace(fusion_config, threshold_high=threshold_high)
    label = decide_label(row, score, fusion_config)
    if label == "uncertain":
        return _fallback_label(matcher)
    return label


def _score_matcher(matcher: Any, pairs: pd.DataFrame) -> tuple[list[float], str]:
    if pairs.empty:
        return [], "skipped_empty_gold_set"
    row_objects = [row for _, row in pairs.iterrows()]
    score_batch = getattr(matcher, "score_batch", None)
    if callable(score_batch):
        scores = score_batch(row_objects)
    else:
        scores = [matcher.score(row) for row in row_objects]
    if scores and all(math.isnan(score) for score in scores):
        return scores, matcher.status().message
    return scores, "ready"


def _summarize_predictions(method: str, frame: pd.DataFrame, *, mode: str, threshold_high: float | None) -> dict[str, object]:
    report = classification_report_df(frame["label"], frame["predicted_label"], labels=EVAL_LABELS)
    macro = report[["precision", "recall", "f1"]].mean()
    exact_precision = float(report.loc[report["label"].eq("exact_duplicate"), "precision"].iloc[0])
    false_merges = frame[frame["false_merge"]]
    return {
        "method": method,
        "mode": mode,
        "eval_split": frame["eval_split"].iloc[0] if frame["eval_split"].nunique() == 1 else "all",
        "threshold_high": threshold_high,
        "pairs": len(frame),
        "macro_precision": float(macro["precision"]),
        "macro_recall": float(macro["recall"]),
        "macro_f1": float(macro["f1"]),
        "exact_duplicate_precision": exact_precision,
        "false_merge_count": int(len(false_merges)),
        "false_merge_rate": float(len(false_merges) / len(frame)) if len(frame) else 0.0,
    }


def _predict_frame(matcher: Any, frame: pd.DataFrame, scores: list[float], *, threshold_high: float | None, mode: str) -> pd.DataFrame:
    output = frame.copy()
    output["method"] = matcher.name
    output["score"] = scores
    output["threshold_high"] = threshold_high
    output["mode"] = mode
    output["predicted_label"] = [
        _predict_from_score(matcher, row, score, threshold_high=threshold_high)
        for (_, row), score in zip(output.iterrows(), scores, strict=False)
    ]
    output["false_merge"] = output["predicted_label"].isin(MERGE_LIKE_LABELS) & output["label"].eq("different_product")
    return output


## Блок кода 6. Подсчёт сходства для всех пар

Эта ячейка прогоняет каждый метод по размеченным парам и сохраняет численные оценки `score`.

Что смотреть в выводе:

- `status=ready` — метод отработал;
- `seconds` — сколько времени занял расчёт.

Для rule-based обычно всё быстро. Для `bi_encoder_zero_shot` может быть дольше, потому что загружается модель и считаются векторы текстов.


In [6]:
scored_methods: dict[str, dict[str, object]] = {}
skipped_methods: list[dict[str, str]] = []

if labeled_pairs.empty:
    display(pd.DataFrame([{"method": "not_available_yet", "status": labeling_status.loc[0, "status"], "seconds": 0.0}]))
else:
    scoring_rows = []
    for matcher in matchers:
        started = time.perf_counter()
        scores, status = _score_matcher(matcher, labeled_pairs)
        elapsed = time.perf_counter() - started
        scoring_rows.append({"method": matcher.name, "status": status, "seconds": round(elapsed, 3)})
        if status != "ready":
            skipped_methods.append({"method": matcher.name, "status": status})
            continue
        scored_methods[matcher.name] = {"matcher": matcher, "scores": scores, "seconds": elapsed}
    display(pd.DataFrame(scoring_rows))
    if skipped_methods:
        display(pd.DataFrame(skipped_methods))


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5282.57it/s]

,method,status,seconds
0,rule_based_fuzzy,ready,0.039
1,bi_encoder_zero_shot,ready,21.303


## Блок кода 7. Быстрая проверка методов на всём размеченном наборе

Эта ячейка показывает грубую картину без честного разделения на подбор порога и финальную проверку.

Эти цифры полезны как быстрый обзор, но финально ориентироваться лучше на таблицу `calibrated / test`, которая появится ниже.

Особенно смотри:

- `macro_f1` — общее качество по двум классам;
- `exact_duplicate_precision` — насколько безопасно метод предлагает точные склейки;
- `false_merge_rate` — доля опасных ошибок.


In [ ]:
default_outputs: dict[str, pd.DataFrame] = {}
default_reports: list[pd.DataFrame] = []
default_summary_rows: list[dict[str, object]] = []

for method, payload in scored_methods.items():
    matcher = payload["matcher"]
    scores = payload["scores"]
    output = _predict_frame(
        matcher,
        labeled_pairs,
        scores,
        threshold_high=_base_fusion_config(matcher).threshold_high,
        mode="default_full_gold_set_sanity",
    )
    default_outputs[method] = output
    default_summary_rows.append(
        _summarize_predictions(
            method,
            output,
            mode="default_full_gold_set_sanity",
            threshold_high=_base_fusion_config(matcher).threshold_high,
        )
    )
    report = classification_report_df(output["label"], output["predicted_label"], labels=EVAL_LABELS)
    report.insert(0, "method", method)
    report.insert(1, "mode", "default_full_gold_set_sanity")
    default_reports.append(report)

if default_summary_rows:
    default_summary_df = pd.DataFrame(default_summary_rows).sort_values("macro_f1", ascending=False)
    default_report_df = pd.concat(default_reports, ignore_index=True)
    display(default_summary_df)
    display(default_report_df)
else:
    display(pd.DataFrame([{"method": "not_available_yet", "mode": "default_full_gold_set_sanity", "pairs": 0}]))


## Блок кода 8. Подбор порога на dev и проверка на test

Это главная ячейка тетрадки.

Что происходит:

1. На `dev` перебираются разные значения `threshold_high`.
2. Для каждого значения считаются метрики.
3. Выбирается порог, который даёт лучший баланс качества и осторожности.
4. Этот выбранный порог проверяется на `test`.

Читать нужно строки `mode=calibrated` и `eval_split=test`. Это самые честные текущие цифры.


In [ ]:
def _threshold_scores_for_split(method: str, payload: dict[str, object], split_name: str) -> tuple[pd.DataFrame, list[float]]:
    split_mask = labeled_pairs["eval_split"].eq(split_name)
    split_frame = labeled_pairs.loc[split_mask].copy()
    split_scores = [score for score, keep in zip(payload["scores"], split_mask, strict=False) if keep]
    return split_frame, split_scores


calibration_rows: list[dict[str, object]] = []
calibrated_outputs: list[pd.DataFrame] = []
calibrated_reports: list[pd.DataFrame] = []
calibrated_summary_rows: list[dict[str, object]] = []

for method, payload in scored_methods.items():
    matcher = payload["matcher"]
    dev_frame, dev_scores = _threshold_scores_for_split(method, payload, "dev")
    if dev_frame.empty:
        continue

    grid_rows = []
    for threshold_high in THRESHOLD_GRID:
        dev_pred = _predict_frame(matcher, dev_frame, dev_scores, threshold_high=threshold_high, mode="calibration_dev")
        grid_rows.append(_summarize_predictions(method, dev_pred, mode="calibration_dev", threshold_high=threshold_high))
    grid = pd.DataFrame(grid_rows)
    target_met = grid[grid["exact_duplicate_precision"].ge(TARGET_EXACT_PRECISION)]
    selection_pool = target_met if not target_met.empty else grid
    selected = selection_pool.sort_values(
        ["macro_f1", "exact_duplicate_precision", "false_merge_rate"],
        ascending=[False, False, True],
    ).iloc[0]
    selected_threshold = float(selected["threshold_high"])
    calibration_rows.append(
        {
            "method": method,
            "selected_threshold_high": selected_threshold,
            "target_exact_precision": TARGET_EXACT_PRECISION,
            "target_met_on_dev": bool(selected["exact_duplicate_precision"] >= TARGET_EXACT_PRECISION),
            "dev_macro_f1": float(selected["macro_f1"]),
            "dev_exact_duplicate_precision": float(selected["exact_duplicate_precision"]),
            "dev_false_merge_rate": float(selected["false_merge_rate"]),
        }
    )

    all_pred = _predict_frame(matcher, labeled_pairs, payload["scores"], threshold_high=selected_threshold, mode="calibrated")
    calibrated_outputs.append(all_pred)
    for split_name in ["dev", "test"]:
        split_pred = all_pred[all_pred["eval_split"].eq(split_name)].copy()
        if split_pred.empty:
            continue
        calibrated_summary_rows.append(
            _summarize_predictions(method, split_pred, mode="calibrated", threshold_high=selected_threshold)
        )
        report = classification_report_df(split_pred["label"], split_pred["predicted_label"], labels=EVAL_LABELS)
        report.insert(0, "method", method)
        report.insert(1, "mode", "calibrated")
        report.insert(2, "eval_split", split_name)
        report.insert(3, "threshold_high", selected_threshold)
        calibrated_reports.append(report)

calibration_df = pd.DataFrame(calibration_rows)
calibrated_summary_df = pd.DataFrame(calibrated_summary_rows)
calibrated_report_df = pd.concat(calibrated_reports, ignore_index=True) if calibrated_reports else pd.DataFrame()
all_calibrated_predictions = pd.concat(calibrated_outputs, ignore_index=True) if calibrated_outputs else pd.DataFrame()

if not calibration_df.empty:
    display(calibration_df)
    display(calibrated_summary_df.sort_values(["eval_split", "macro_f1"], ascending=[True, False]))
    display(calibrated_report_df)
else:
    display(pd.DataFrame([{"status": "no_methods_available_for_calibration"}]))


## Блок кода 9. Матрицы ошибок и примеры опасных ошибок

Эта ячейка показывает, какие именно классы метод путает.

В матрице ошибок строки — правильная ручная метка, колонки — предсказание метода. Числа на диагонали — правильные ответы. Всё вне диагонали — ошибки.

Самая опасная ячейка: строка `different_product`, колонка `exact_duplicate`. Это случаи, где разные товары метод пытается связать.

Ниже выводятся конкретные пары с такими опасными ошибками. Их полезно читать глазами: они объясняют, почему простой метод ошибается.


In [ ]:
if all_calibrated_predictions.empty:
    print("Confusion matrices и false-merge examples появятся после запуска хотя бы одного метода на размеченных парах.")
else:
    for method in all_calibrated_predictions["method"].drop_duplicates():
        for split_name in ["dev", "test"]:
            part = all_calibrated_predictions[
                all_calibrated_predictions["method"].eq(method)
                & all_calibrated_predictions["eval_split"].eq(split_name)
            ]
            if part.empty:
                continue
            print(f"Confusion matrix: {method} / calibrated / {split_name}")
            display(confusion_matrix_df(part["label"], part["predicted_label"], labels=EVAL_LABELS))

    false_merge_examples = all_calibrated_predictions[
        all_calibrated_predictions["eval_split"].eq("test") & all_calibrated_predictions["false_merge"]
    ].copy()
    if false_merge_examples.empty:
        print("На held-out test false-merge примеров нет.")
    else:
        print(f"False-merge examples on held-out test: {len(false_merge_examples)}")
        columns = [
            "method",
            "label",
            "predicted_label",
            "score",
            "threshold_high",
            "title_a",
            "title_b",
            "brand_a",
            "brand_b",
            "unit_amount_a",
            "unit_amount_b",
            "total_amount_a",
            "total_amount_b",
            "multipack_count_a",
            "multipack_count_b",
        ]
        display(false_merge_examples[[column for column in columns if column in false_merge_examples.columns]].head(20))


## Блок кода 10. Сохранение результатов для следующих тетрадок

Эта ячейка сохраняет таблицы, которые потом читает тетрадка `04_clustering_resolution.ipynb`.

Сохраняются:

- `matching_summary_sauces.csv` — короткая таблица с метриками методов;
- `matching_predictions_sauces.csv` — все предсказания по парам;
- `matching_false_merges_sauces.csv` — отдельный список опасных ошибок.

Эти CSV лежат в `research/dedup/data/` и не коммитятся.


In [10]:
summary_frames = []
if default_summary_rows:
    summary_frames.append(pd.DataFrame(default_summary_rows))
if calibrated_summary_rows:
    summary_frames.append(pd.DataFrame(calibrated_summary_rows))

if summary_frames:
    summary_export = pd.concat(summary_frames, ignore_index=True)
    summary_export.to_csv(SUMMARY_PATH, index=False)
    print(f"Saved summary: {SUMMARY_PATH} ({len(summary_export)} rows)")
else:
    summary_export = pd.DataFrame()
    print("Summary export skipped: no methods available.")

if not all_calibrated_predictions.empty:
    all_calibrated_predictions.to_csv(PREDICTIONS_PATH, index=False)
    print(f"Saved predictions: {PREDICTIONS_PATH} ({len(all_calibrated_predictions)} rows)")
    false_merges_export = all_calibrated_predictions[all_calibrated_predictions["false_merge"]].copy()
    false_merges_export.to_csv(FALSE_MERGES_PATH, index=False)
    print(f"Saved false merges: {FALSE_MERGES_PATH} ({len(false_merges_export)} rows)")
else:
    print("Prediction export skipped: no calibrated outputs.")


Saved summary: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_summary_sauces.csv (6 rows)
Saved predictions: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_predictions_sauces.csv (758 rows)
Saved false merges: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_false_merges_sauces.csv (104 rows)


## Что делать после этой тетрадки

Если результат `rule_based_fuzzy` и `bi_encoder_zero_shot` слабый, это нормально для первого прогона. Эта тетрадка нужна не для финальной победы, а чтобы увидеть нижнюю планку и понять, где методы ошибаются.

Главный вывод сейчас: похожесть текста сама по себе часто путает разные вкусы и типы соусов. Следующий разумный шаг — добавить более сильную проверку пары, например cross-encoder или отдельную проверку спорных случаев.


## Новая итерация: добавляем готовый cross-encoder

Предыдущие блоки показали важную проблему: простая похожесть названий и обычные векторы часто путают похожие, но разные товары.

Теперь добавляем следующий метод из архитектуры: cross-encoder. Он читает пару товаров вместе: товар A и товар B одновременно. Поэтому он теоретически должен лучше замечать различия вроде `сырный` против `барбекю` или `сальса` против `сладкий чили`.

Пока это не обученная на наших данных модель, а готовая модель из `sentence-transformers`. Поэтому это промежуточный опыт: проверяем, помогает ли более внимательное сравнение пары даже без дообучения.


## Блок кода 11. Настройки cross-encoder

Эта ячейка добавляет новый метод, но не трогает старые результаты выше.

Что важно:

- `DEDUP_RUN_CROSS_ENCODER=0` можно поставить, если нужно временно пропустить этот блок.
- `DEDUP_CROSS_ENCODER_MODEL` теперь принимает alias из model registry или прямой Hugging Face model id.
- По умолчанию используется alias `cross_encoder_mmarco`, который указывает на `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`.
- Скачивание и повторное использование модели управляется через `ModelManager`: кэш `DEDUP_MODEL_CACHE_DIR`, offline-флаг `DEDUP_MODEL_LOCAL_ONLY=1`.

Если модель ещё не скачана, первый запуск может занять время. Если интернет недоступен, включи offline-режим только после предварительного прогрева кэша.


In [ ]:
from research.dedup import CrossEncoderMatcher
from research.dedup.matchers.cross_encoder import CrossEncoderConfig

RUN_CROSS_ENCODER = os.environ.get("DEDUP_RUN_CROSS_ENCODER", "1") == "1"
CROSS_ENCODER_MODEL = os.environ.get("DEDUP_CROSS_ENCODER_MODEL", "cross_encoder_mmarco")
CROSS_ENCODER_MODEL_SPEC = MODEL_MANAGER.resolve(CROSS_ENCODER_MODEL, backend=CROSS_ENCODER_BACKEND)
CROSS_ENCODER_BATCH_SIZE = int(
    os.environ.get("DEDUP_CROSS_ENCODER_BATCH_SIZE", str(CROSS_ENCODER_MODEL_SPEC.batch_size or 16))
)

print(f"Run cross-encoder: {RUN_CROSS_ENCODER}")
print(f"Cross-encoder model alias/input: {CROSS_ENCODER_MODEL}")
print(f"Cross-encoder model id: {CROSS_ENCODER_MODEL_SPEC.model_name}")
print(f"Cross-encoder batch size: {CROSS_ENCODER_BATCH_SIZE}")
print(f"Model cache dir: {MODEL_MANAGER.cache_dir}")
print(f"Local-only model loading: {MODEL_MANAGER.local_files_only}")


## Блок кода 12. Запуск cross-encoder на тех же парах

Эта ячейка считает `score` для каждой размеченной пары.

`score` здесь означает: насколько готовая модель считает пару подходящей для связи. У этой модели score не обязан быть от 0 до 1: важен не абсолютный смысл числа, а то, как оно разделяет хорошие и плохие пары. Потом этот score проходит через ту же простую логику с весом, фасовкой и брендом, чтобы получить один из трёх классов.

Мы используем те же `dev` и `test`, что выше. Это важно: новый метод сравнивается на той же отложенной части, а не на новых случайных строках.


In [12]:
cross_encoder_payload: dict[str, object] | None = None
cross_encoder_status_rows: list[dict[str, object]] = []

if not RUN_CROSS_ENCODER:
    cross_encoder_status_rows.append({"method": "cross_encoder_zero_shot", "status": "skipped_by_env", "seconds": 0.0})
elif labeled_pairs.empty:
    cross_encoder_status_rows.append({"method": "cross_encoder_zero_shot", "status": "skipped_empty_gold_set", "seconds": 0.0})
else:
    cross_encoder = CrossEncoderMatcher(
        CrossEncoderConfig(
            model_name=CROSS_ENCODER_MODEL,
            batch_size=CROSS_ENCODER_BATCH_SIZE,
        )
    )
    started = time.perf_counter()
    cross_scores, cross_status = _score_matcher(cross_encoder, labeled_pairs)
    elapsed = time.perf_counter() - started
    cross_encoder_status_rows.append(
        {"method": cross_encoder.name, "status": cross_status, "seconds": round(elapsed, 3)}
    )
    if cross_status == "ready":
        cross_encoder_payload = {"matcher": cross_encoder, "scores": cross_scores, "seconds": elapsed}

cross_encoder_status_df = pd.DataFrame(cross_encoder_status_rows)
display(cross_encoder_status_df)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6401.09it/s]

,method,status,seconds
0,cross_encoder_zero_shot,ready,27.663


## Блок кода 13. Калибровка cross-encoder и сравнение с предыдущими методами

Эта ячейка подбирает порог для cross-encoder на `dev`, потом проверяет его на `test`. Порог подбирается по реальному диапазону score этой модели, а не по фиксированной шкале 0..1.

Читать нужно так же, как выше:

- `macro_f1` — общая картина по двум классам;
- `exact_duplicate_precision` — насколько безопасны точные склейки;
- `false_merge_rate` — доля опасных ошибок.

Главный вопрос: стало ли меньше опасных склеек по сравнению с `rule_based_fuzzy` и `bi_encoder_zero_shot`.


In [ ]:
cross_default_summary_rows: list[dict[str, object]] = []
cross_calibration_rows: list[dict[str, object]] = []
cross_calibrated_summary_rows: list[dict[str, object]] = []
cross_calibrated_reports: list[pd.DataFrame] = []
cross_calibrated_predictions = pd.DataFrame()

if cross_encoder_payload is None:
    display(pd.DataFrame([{"status": "cross_encoder_not_available"}]))
else:
    matcher = cross_encoder_payload["matcher"]
    scores = cross_encoder_payload["scores"]

    default_output = _predict_frame(
        matcher,
        labeled_pairs,
        scores,
        threshold_high=_base_fusion_config(matcher).threshold_high,
        mode="default_full_gold_set_sanity",
    )
    cross_default_summary_rows.append(
        _summarize_predictions(
            matcher.name,
            default_output,
            mode="default_full_gold_set_sanity",
            threshold_high=_base_fusion_config(matcher).threshold_high,
        )
    )

    dev_frame, dev_scores = _threshold_scores_for_split(matcher.name, cross_encoder_payload, "dev")
    score_series = pd.Series(dev_scores).dropna()
    if score_series.empty:
        cross_threshold_grid = THRESHOLD_GRID
    else:
        quantile_points = [idx / 100 for idx in range(5, 100, 5)]
        cross_threshold_grid = sorted(
            {
                round(float(value), 6)
                for value in score_series.quantile(quantile_points).tolist()
                + [score_series.min(), score_series.max(), _base_fusion_config(matcher).threshold_high]
            }
        )

    grid_rows = []
    for threshold_high in cross_threshold_grid:
        dev_pred = _predict_frame(matcher, dev_frame, dev_scores, threshold_high=threshold_high, mode="calibration_dev")
        grid_rows.append(_summarize_predictions(matcher.name, dev_pred, mode="calibration_dev", threshold_high=threshold_high))
    grid = pd.DataFrame(grid_rows)
    target_met = grid[grid["exact_duplicate_precision"].ge(TARGET_EXACT_PRECISION)]
    selection_pool = target_met if not target_met.empty else grid
    selected = selection_pool.sort_values(
        ["macro_f1", "exact_duplicate_precision", "false_merge_rate"],
        ascending=[False, False, True],
    ).iloc[0]
    selected_threshold = float(selected["threshold_high"])
    cross_calibration_rows.append(
        {
            "method": matcher.name,
            "selected_threshold_high": selected_threshold,
            "target_exact_precision": TARGET_EXACT_PRECISION,
            "target_met_on_dev": bool(selected["exact_duplicate_precision"] >= TARGET_EXACT_PRECISION),
            "dev_macro_f1": float(selected["macro_f1"]),
            "dev_exact_duplicate_precision": float(selected["exact_duplicate_precision"]),
            "dev_false_merge_rate": float(selected["false_merge_rate"]),
        }
    )

    cross_calibrated_predictions = _predict_frame(
        matcher,
        labeled_pairs,
        scores,
        threshold_high=selected_threshold,
        mode="calibrated",
    )
    for split_name in ["dev", "test"]:
        split_pred = cross_calibrated_predictions[cross_calibrated_predictions["eval_split"].eq(split_name)].copy()
        cross_calibrated_summary_rows.append(
            _summarize_predictions(matcher.name, split_pred, mode="calibrated", threshold_high=selected_threshold)
        )
        report = classification_report_df(split_pred["label"], split_pred["predicted_label"], labels=EVAL_LABELS)
        report.insert(0, "method", matcher.name)
        report.insert(1, "mode", "calibrated")
        report.insert(2, "eval_split", split_name)
        report.insert(3, "threshold_high", selected_threshold)
        cross_calibrated_reports.append(report)

    combined_summary = pd.concat(
        [
            summary_export,
            pd.DataFrame(cross_default_summary_rows),
            pd.DataFrame(cross_calibrated_summary_rows),
        ],
        ignore_index=True,
    )
    display(pd.DataFrame(cross_calibration_rows))
    display(combined_summary.sort_values(["mode", "eval_split", "macro_f1"], ascending=[True, True, False]))
    display(pd.concat(cross_calibrated_reports, ignore_index=True))


Если cross-encoder полезен, мы хотим увидеть меньше строк, где настоящий `different_product` попал в `exact_duplicate`.


In [ ]:
if cross_calibrated_predictions.empty:
    print("Cross-encoder confusion matrices появятся после успешного запуска модели.")
else:
    for split_name in ["dev", "test"]:
        part = cross_calibrated_predictions[cross_calibrated_predictions["eval_split"].eq(split_name)].copy()
        print(f"Confusion matrix: cross_encoder_zero_shot / calibrated / {split_name}")
        display(confusion_matrix_df(part["label"], part["predicted_label"], labels=EVAL_LABELS))

    cross_false_merges = cross_calibrated_predictions[
        cross_calibrated_predictions["eval_split"].eq("test") & cross_calibrated_predictions["false_merge"]
    ].copy()
    print(f"Cross-encoder false-merge examples on held-out test: {len(cross_false_merges)}")
    columns = [
        "label",
        "predicted_label",
        "score",
        "threshold_high",
        "title_a",
        "title_b",
        "brand_a",
        "brand_b",
        "unit_amount_a",
        "unit_amount_b",
        "multipack_count_a",
        "multipack_count_b",
    ]
    display(cross_false_merges[[column for column in columns if column in cross_false_merges.columns]].head(20))


## Блок кода 15. Обновление CSV после добавления cross-encoder

Эта ячейка дописывает результаты cross-encoder в те же CSV, которые потом читают `04` и `05`.

Если cross-encoder не запустился, старые CSV остаются как были. Если запустился, в файлах появится ещё один метод: `cross_encoder_zero_shot`.


In [15]:
if cross_calibrated_predictions.empty:
    updated_summary = summary_export.copy()
    updated_predictions = all_calibrated_predictions.copy()
    updated_false_merges = updated_predictions[updated_predictions["false_merge"]].copy() if not updated_predictions.empty else pd.DataFrame()
    print("Cross-encoder export skipped: no calibrated predictions.")
else:
    updated_summary = pd.concat(
        [
            summary_export,
            pd.DataFrame(cross_default_summary_rows),
            pd.DataFrame(cross_calibrated_summary_rows),
        ],
        ignore_index=True,
    )
    updated_predictions = pd.concat([all_calibrated_predictions, cross_calibrated_predictions], ignore_index=True)
    updated_false_merges = updated_predictions[updated_predictions["false_merge"]].copy()

    updated_summary.to_csv(SUMMARY_PATH, index=False)
    updated_predictions.to_csv(PREDICTIONS_PATH, index=False)
    updated_false_merges.to_csv(FALSE_MERGES_PATH, index=False)

    print(f"Saved updated summary: {SUMMARY_PATH} ({len(updated_summary)} rows)")
    print(f"Saved updated predictions: {PREDICTIONS_PATH} ({len(updated_predictions)} rows)")
    print(f"Saved updated false merges: {FALSE_MERGES_PATH} ({len(updated_false_merges)} rows)")

Saved updated summary: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_summary_sauces.csv (9 rows)
Saved updated predictions: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_predictions_sauces.csv (1137 rows)
Saved updated false merges: /Users/exoldoff/.codex/worktrees/13ef/mpstats/research/dedup/data/matching_false_merges_sauces.csv (182 rows)


## Что мы проверяем этой новой итерацией

Эта секция отвечает на конкретный вопрос: помогает ли готовый cross-encoder как более внимательная проверка пары.

Если он заметно снижает false merges на `test`, это сильный аргумент двигаться в сторону cross-encoder rerank.

Если он не помогает достаточно, это тоже нормальный результат: тогда показываем жюри, что готовой модели мало, и нужен следующий шаг — дообучение на наших парах или отдельный LLM-judge для спорных случаев.


## Общий бенчмарк всех matching-моделей

Эта секция сравнивает текущие baseline-методы и выбранные reranker-модели на одном и том же наборе размеченных пар:

- `rule_based_fuzzy`;
- `bi_encoder_zero_shot`;
- `cross_encoder_zero_shot`;
- `BAAI/bge-reranker-v2-m3` по умолчанию;
- опционально `Qwen/Qwen3-Reranker-4B`, `Qwen/Qwen3-Reranker-0.6B` и `jinaai/jina-reranker-v3`.

Запуск обычный: меняете настройки в следующей ячейке и жмёте Run. Никакие переменные окружения для включения блока не нужны.

Первый запуск может быть долгим именно на скачивании модели из Hugging Face. `Qwen/Qwen3-Reranker-4B` тяжёлый и на Mac может не помещаться в MPS, поэтому registry запускает его на CPU. Если нужен быстрый Qwen-smoke, попробуйте alias `qwen3_0_6b`.

## Блок кода 16. Настройки общего бенчмарка

Главное место для настройки моделей — верх следующей code-ячейки, блок `НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ`.

Туда вписываются новые reranker-модели, которые добавляются к уже посчитанным выше baseline-методам. Старые методы (`rule_based_fuzzy`, `bi_encoder_zero_shot`, `cross_encoder_zero_shot`) перечислять не нужно: они подтягиваются из предыдущих блоков этой же тетрадки автоматически.

Что менять чаще всего:

- `MY_RERANKER_MODELS` — список моделей для запуска. Можно писать короткие alias-ы (`"bge_m3"`, `"qwen3_4b"`, `"jina_v3"`) или полные Hugging Face model ids (`"BAAI/bge-reranker-v2-m3"`).
- `MY_RERANKER_MAX_PAIRS` — размер быстрого среза. `120` удобно для первого прогона, `0` означает весь размеченный gold-set.
- `MY_CUSTOM_RERANKER_BACKEND` — backend для model id, которого ещё нет в registry. Для обычных Hugging Face cross-encoder моделей оставляйте `CROSS_ENCODER_BACKEND`.

Переменные окружения `DEDUP_RERANKER_BENCHMARK_MODELS`, `DEDUP_RERANKER_BENCHMARK_MAX_PAIRS` и `DEDUP_RERANKER_BENCHMARK_BACKEND` нужны только для запуска из терминала; если они заданы, они переопределяют значения из code-ячейки.

После запуска ячейка показывает таблицу: что вы попросили, какой alias реально резолвится, какой model id будет скачан, какой backend используется и где лежит cache.


In [ ]:
from research.dedup import CrossEncoderMatcher, JinaRerankerMatcher
from research.dedup.matchers.cross_encoder import CrossEncoderConfig
from research.dedup.matchers.jina_reranker import JinaRerankerConfig

# === НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ ===
# Сюда вписывайте свои reranker-модели для общего benchmark.
# Можно писать короткие aliases: "bge_m3", "qwen3_4b", "qwen3_0_6b", "jina_v3".
# Можно писать полные Hugging Face ids: "BAAI/bge-reranker-v2-m3", "Qwen/Qwen3-Reranker-4B".
MY_RERANKER_MODELS = [
    "bge_m3",
    # "qwen3_4b",  # 4B грузится на CPU, чтобы не падать с MPS out of memory.
    # "qwen3_0_6b",  # более лёгкий Qwen для быстрого smoke-прогона.
    # "jina_v3",
    # "cross-encoder/ms-marco-MiniLM-L6-v2",  # пример своего cross-encoder id
]

# 120 = быстрый пробный срез. Для первого CPU-запуска Qwen-4B можно поставить 12-30. 0 = весь размеченный gold-set.
MY_RERANKER_MAX_PAIRS = 120

# Для своих Hugging Face cross-encoder ids оставляйте CROSS_ENCODER_BACKEND.
# Для Jina-like моделей с методом .rerank используйте TRANSFORMERS_AUTO_MODEL_BACKEND.
# Если хотите запрещать неизвестные ids, поставьте None.
MY_CUSTOM_RERANKER_BACKEND = CROSS_ENCODER_BACKEND
# === КОНЕЦ НАСТРОЕК ПОЛЬЗОВАТЕЛЯ ===


_reranker_models_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_MODELS")
if _reranker_models_env is None:
    RERANKER_BENCHMARK_MODELS = [str(model).strip() for model in MY_RERANKER_MODELS if str(model).strip()]
    RERANKER_BENCHMARK_MODELS_SOURCE = "notebook: MY_RERANKER_MODELS"
else:
    RERANKER_BENCHMARK_MODELS = [model.strip() for model in _reranker_models_env.split(",") if model.strip()]
    RERANKER_BENCHMARK_MODELS_SOURCE = "env: DEDUP_RERANKER_BENCHMARK_MODELS"

_reranker_max_pairs_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_MAX_PAIRS")
if _reranker_max_pairs_env is None:
    RERANKER_BENCHMARK_MAX_PAIRS = int(MY_RERANKER_MAX_PAIRS)
    RERANKER_BENCHMARK_MAX_PAIRS_SOURCE = "notebook: MY_RERANKER_MAX_PAIRS"
else:
    RERANKER_BENCHMARK_MAX_PAIRS = int(_reranker_max_pairs_env)
    RERANKER_BENCHMARK_MAX_PAIRS_SOURCE = "env: DEDUP_RERANKER_BENCHMARK_MAX_PAIRS"

_custom_backend_env = os.environ.get("DEDUP_RERANKER_BENCHMARK_BACKEND")
CUSTOM_RERANKER_BACKEND = _custom_backend_env.strip() if _custom_backend_env is not None else MY_CUSTOM_RERANKER_BACKEND
if isinstance(CUSTOM_RERANKER_BACKEND, str) and CUSTOM_RERANKER_BACKEND.strip().lower() in {"", "none", "null"}:
    CUSTOM_RERANKER_BACKEND = None

RERANKER_BENCHMARK_SUMMARY_PATH = DATA_DIR / "reranker_benchmark_summary_sauces.csv"
RERANKER_BENCHMARK_PREDICTIONS_PATH = DATA_DIR / "reranker_benchmark_predictions_sauces.csv"
ALL_MODEL_BENCHMARK_SUMMARY_PATH = DATA_DIR / "all_model_benchmark_summary_sauces.csv"
ALL_MODEL_BENCHMARK_PREDICTIONS_PATH = DATA_DIR / "all_model_benchmark_predictions_sauces.csv"

print(f"Models source: {RERANKER_BENCHMARK_MODELS_SOURCE}")
print(f"Requested reranker models: {RERANKER_BENCHMARK_MODELS or '[]'}")
print(f"Max pairs source: {RERANKER_BENCHMARK_MAX_PAIRS_SOURCE}; value={RERANKER_BENCHMARK_MAX_PAIRS or 'all'}")
print(f"Custom model backend: {CUSTOM_RERANKER_BACKEND or 'disabled'}")


def _fusion_from_model_spec(spec):
    threshold_high = spec.fusion_threshold_high if spec.fusion_threshold_high is not None else 0.5
    threshold_low = spec.fusion_threshold_low if spec.fusion_threshold_low is not None else 0.2
    return FusionConfig(threshold_high=threshold_high, threshold_low=threshold_low)


def _resolve_reranker_specs(model_inputs, *, custom_backend=None):
    specs = []
    errors = []
    input_by_alias = {}
    allowed_backends = {CROSS_ENCODER_BACKEND, TRANSFORMERS_AUTO_MODEL_BACKEND}
    if custom_backend not in allowed_backends | {None}:
        errors.append({"model_input": "MY_CUSTOM_RERANKER_BACKEND", "error": f"unsupported custom backend: {custom_backend}"})
        custom_backend = None

    for model_input in model_inputs:
        model_input = str(model_input).strip()
        if not model_input:
            continue
        try:
            spec = MODEL_MANAGER.resolve(model_input)
        except Exception as exc:
            if custom_backend is None:
                errors.append({
                    "model_input": model_input,
                    "error": f"{exc}; set MY_CUSTOM_RERANKER_BACKEND for custom model ids",
                })
                continue
            try:
                spec = MODEL_MANAGER.resolve(model_input, backend=custom_backend)
            except Exception as fallback_exc:
                errors.append({
                    "model_input": model_input,
                    "error": f"{exc}; custom backend {custom_backend}: {fallback_exc}",
                })
                continue
        if spec.backend not in allowed_backends:
            errors.append({"model_input": model_input, "error": f"unsupported backend for reranker benchmark: {spec.backend}"})
            continue
        specs.append(spec)
        input_by_alias[spec.alias] = model_input
    return specs, errors, input_by_alias


reranker_model_specs, reranker_model_errors, reranker_model_inputs = _resolve_reranker_specs(
    RERANKER_BENCHMARK_MODELS,
    custom_backend=CUSTOM_RERANKER_BACKEND,
)

benchmark_config_rows = [
    {
        "input": reranker_model_inputs.get(spec.alias, spec.alias),
        "alias": spec.alias,
        "method": spec.method_name or spec.alias,
        "backend": spec.backend,
        "model": spec.model_name,
        "batch_size": spec.batch_size or "",
        "device": spec.device or "auto",
        "documents_per_query": spec.documents_per_query or "",
        "max_pairs": RERANKER_BENCHMARK_MAX_PAIRS or "all",
        "cache_dir": str(MODEL_MANAGER.cache_dir),
        "local_only": MODEL_MANAGER.local_files_only,
    }
    for spec in reranker_model_specs
]
display(pd.DataFrame(benchmark_config_rows))
if reranker_model_errors:
    display(pd.DataFrame(reranker_model_errors))

reranker_benchmark_matchers = []
for spec in reranker_model_specs:
    method_name = spec.method_name or spec.alias
    if spec.backend == CROSS_ENCODER_BACKEND:
        reranker_benchmark_matchers.append(
            CrossEncoderMatcher(
                CrossEncoderConfig(
                    model_name=spec.alias,
                    method_name=method_name,
                    batch_size=spec.batch_size or 1,
                    device=spec.device,
                    trust_remote_code=spec.trust_remote_code,
                    prompts=spec.prompts,
                    default_prompt_name=spec.default_prompt_name,
                    fusion=_fusion_from_model_spec(spec),
                )
            )
        )
    elif spec.backend == TRANSFORMERS_AUTO_MODEL_BACKEND:
        reranker_benchmark_matchers.append(
            JinaRerankerMatcher(
                JinaRerankerConfig(
                    model_name=spec.alias,
                    method_name=method_name,
                    documents_per_query=spec.documents_per_query or 8,
                    trust_remote_code=spec.trust_remote_code,
                    fusion=_fusion_from_model_spec(spec),
                )
            )
        )

if not reranker_benchmark_matchers:
    display(pd.DataFrame([{
        "status": "no_models_selected",
        "hint": "add models to MY_RERANKER_MODELS, for example bge_m3, qwen3_4b or jina_v3",
    }]))
else:
    display(pd.DataFrame([
        {"method": matcher.name, "status": matcher.status().message}
        for matcher in reranker_benchmark_matchers
    ]))


## Блок кода 17. Запуск новых reranker-моделей

Эта ячейка считает score только для новых тяжёлых моделей из `MY_RERANKER_MODELS` после возможного env override. Старые методы выше уже посчитаны, поэтому здесь они не запускаются повторно.

Если `MY_RERANKER_MAX_PAIRS > 0`, берётся небольшой сбалансированный срез по `dev/test` и классам. Такой срез годится для проверки, что модель запускается. Для финального выбора лучшего решения поставьте `MY_RERANKER_MAX_PAIRS = 0` в предыдущей ячейке.


In [ ]:
def _with_benchmark_pair_key(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    if {"raw_record_id_a", "raw_record_id_b"}.issubset(output.columns):
        left_values = output["raw_record_id_a"].astype(str)
        right_values = output["raw_record_id_b"].astype(str)
    else:
        left_values = output.get("title_a", pd.Series([""] * len(output))).astype(str)
        right_values = output.get("title_b", pd.Series([""] * len(output))).astype(str)
    output["benchmark_pair_key"] = [
        " || ".join(sorted([left, right]))
        for left, right in zip(left_values, right_values, strict=False)
    ]
    return output


def _benchmark_frame(frame: pd.DataFrame, max_pairs: int) -> pd.DataFrame:
    if frame.empty or max_pairs <= 0 or len(frame) <= max_pairs:
        return _with_benchmark_pair_key(frame)
    ordered = frame.copy()
    ordered["_round_robin_order"] = ordered.groupby(["eval_split", "label"]).cumcount()
    sampled = (
        ordered.sort_values(["_round_robin_order", "eval_split", "label"])
        .head(max_pairs)
        .drop(columns=["_round_robin_order"])
        .sort_index()
    )
    return _with_benchmark_pair_key(sampled.reset_index(drop=True))


reranker_benchmark_pairs = _benchmark_frame(labeled_pairs, RERANKER_BENCHMARK_MAX_PAIRS)
reranker_benchmark_payloads: dict[str, dict[str, object]] = {}
reranker_benchmark_status_rows: list[dict[str, object]] = []

if reranker_benchmark_pairs.empty:
    print("Benchmark skipped: gold-set is empty.")
else:
    print(f"Benchmark pairs: {len(reranker_benchmark_pairs)} / {len(labeled_pairs)}")
    for matcher in reranker_benchmark_matchers:
        started = time.perf_counter()
        scores, status = _score_matcher(matcher, reranker_benchmark_pairs)
        elapsed = time.perf_counter() - started
        reranker_benchmark_status_rows.append(
            {
                "method": matcher.name,
                "model": getattr(matcher.config, "model_name", ""),
                "status": status,
                "seconds": round(elapsed, 3),
                "pairs": len(reranker_benchmark_pairs),
                "seconds_per_pair": round(elapsed / len(reranker_benchmark_pairs), 4) if len(reranker_benchmark_pairs) else 0.0,
            }
        )
        if status == "ready":
            reranker_benchmark_payloads[matcher.name] = {
                "matcher": matcher,
                "scores": scores,
                "seconds": elapsed,
                "pairs": reranker_benchmark_pairs,
            }

if reranker_benchmark_status_rows:
    display(pd.DataFrame(reranker_benchmark_status_rows))

## Блок кода 18. Общая таблица качества всех методов

Здесь собирается одна итоговая таблица для выбора лучшего решения.

Что происходит:

1. Новые reranker-модели калибруют свой порог на `dev` выбранного benchmark-среза.
2. Уже посчитанные выше методы (`rule_based_fuzzy`, `bi_encoder_zero_shot`, `cross_encoder_zero_shot`) берутся из предыдущих результатов и фильтруются на тот же benchmark-срез.
3. Все методы сохраняются в общий CSV `all_model_benchmark_summary_sauces.csv`.

Для финального решения смотрим строки `all_model_benchmark_calibrated / test`.

In [ ]:
def _dynamic_threshold_grid(scores: list[float], base_threshold: float) -> list[float]:
    score_series = pd.Series(scores).dropna()
    if score_series.empty:
        return THRESHOLD_GRID
    quantile_points = [idx / 100 for idx in range(5, 100, 5)]
    return sorted(
        {
            round(float(value), 6)
            for value in score_series.quantile(quantile_points).tolist()
            + [score_series.min(), score_series.max(), base_threshold]
        }
    )


def _scores_for_frame_split(frame: pd.DataFrame, scores: list[float], split_name: str) -> tuple[pd.DataFrame, list[float]]:
    split_mask = frame["eval_split"].eq(split_name)
    split_frame = frame.loc[split_mask].copy()
    split_scores = [score for score, keep in zip(scores, split_mask, strict=False) if keep]
    return split_frame, split_scores


def _threshold_for_summary(frame: pd.DataFrame) -> float | None:
    if "threshold_high" not in frame.columns:
        return None
    values = pd.Series(frame["threshold_high"]).dropna().unique()
    if len(values) == 1:
        return float(values[0])
    return None


def _summaries_for_splits(frame: pd.DataFrame, *, mode: str) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    if frame.empty:
        return rows
    for method in frame["method"].drop_duplicates():
        method_frame = frame[frame["method"].eq(method)].copy()
        for split_name in ["dev", "test"]:
            split_frame = method_frame[method_frame["eval_split"].eq(split_name)].copy()
            if split_frame.empty:
                continue
            rows.append(
                _summarize_predictions(
                    method,
                    split_frame,
                    mode=mode,
                    threshold_high=_threshold_for_summary(split_frame),
                )
            )
    return rows


def _existing_predictions_for_benchmark() -> pd.DataFrame:
    existing = globals().get("updated_predictions", pd.DataFrame())
    if not isinstance(existing, pd.DataFrame) or existing.empty:
        frames = []
        if isinstance(all_calibrated_predictions, pd.DataFrame) and not all_calibrated_predictions.empty:
            frames.append(all_calibrated_predictions)
        if "cross_calibrated_predictions" in globals() and isinstance(cross_calibrated_predictions, pd.DataFrame):
            if not cross_calibrated_predictions.empty:
                frames.append(cross_calibrated_predictions)
        existing = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if existing.empty or reranker_benchmark_pairs.empty:
        return pd.DataFrame()

    benchmark_keys = set(reranker_benchmark_pairs["benchmark_pair_key"])
    keyed = _with_benchmark_pair_key(existing)
    subset = keyed[keyed["benchmark_pair_key"].isin(benchmark_keys)].copy()
    if subset.empty:
        return subset
    subset["source_mode"] = subset.get("mode", "calibrated")
    subset["mode"] = "all_model_benchmark_calibrated"
    subset["benchmark_source"] = "existing_method"
    return subset


benchmark_default_summary_rows: list[dict[str, object]] = []
benchmark_calibration_rows: list[dict[str, object]] = []
benchmark_calibrated_summary_rows: list[dict[str, object]] = []
benchmark_calibrated_reports: list[pd.DataFrame] = []
benchmark_calibrated_outputs: list[pd.DataFrame] = []

for method, payload in reranker_benchmark_payloads.items():
    matcher = payload["matcher"]
    frame = payload["pairs"]
    scores = payload["scores"]
    base_threshold = _base_fusion_config(matcher).threshold_high

    default_output = _predict_frame(
        matcher,
        frame,
        scores,
        threshold_high=base_threshold,
        mode="reranker_benchmark_default",
    )
    benchmark_default_summary_rows.append(
        _summarize_predictions(method, default_output, mode="reranker_benchmark_default", threshold_high=base_threshold)
    )

    dev_frame, dev_scores = _scores_for_frame_split(frame, scores, "dev")
    if dev_frame.empty:
        continue
    grid_rows = []
    for threshold_high in _dynamic_threshold_grid(dev_scores, base_threshold):
        dev_pred = _predict_frame(matcher, dev_frame, dev_scores, threshold_high=threshold_high, mode="reranker_benchmark_dev")
        grid_rows.append(_summarize_predictions(method, dev_pred, mode="reranker_benchmark_dev", threshold_high=threshold_high))
    grid = pd.DataFrame(grid_rows)
    target_met = grid[grid["exact_duplicate_precision"].ge(TARGET_EXACT_PRECISION)]
    selection_pool = target_met if not target_met.empty else grid
    selected = selection_pool.sort_values(
        ["macro_f1", "exact_duplicate_precision", "false_merge_rate"],
        ascending=[False, False, True],
    ).iloc[0]
    selected_threshold = float(selected["threshold_high"])
    benchmark_calibration_rows.append(
        {
            "method": method,
            "model": getattr(matcher.config, "model_name", ""),
            "selected_threshold_high": selected_threshold,
            "target_exact_precision": TARGET_EXACT_PRECISION,
            "target_met_on_dev": bool(selected["exact_duplicate_precision"] >= TARGET_EXACT_PRECISION),
            "dev_macro_f1": float(selected["macro_f1"]),
            "dev_exact_duplicate_precision": float(selected["exact_duplicate_precision"]),
            "dev_false_merge_rate": float(selected["false_merge_rate"]),
        }
    )

    calibrated = _predict_frame(matcher, frame, scores, threshold_high=selected_threshold, mode="all_model_benchmark_calibrated")
    calibrated["source_mode"] = "reranker_benchmark_calibrated"
    calibrated["benchmark_source"] = "new_reranker"
    benchmark_calibrated_outputs.append(calibrated)
    for split_name in ["dev", "test"]:
        split_pred = calibrated[calibrated["eval_split"].eq(split_name)].copy()
        if split_pred.empty:
            continue
        benchmark_calibrated_summary_rows.append(
            _summarize_predictions(method, split_pred, mode="all_model_benchmark_calibrated", threshold_high=selected_threshold)
        )
        report = classification_report_df(split_pred["label"], split_pred["predicted_label"], labels=EVAL_LABELS)
        report.insert(0, "method", method)
        report.insert(1, "mode", "all_model_benchmark_calibrated")
        report.insert(2, "eval_split", split_name)
        report.insert(3, "threshold_high", selected_threshold)
        benchmark_calibrated_reports.append(report)

benchmark_summary_frames = []
if benchmark_default_summary_rows:
    benchmark_summary_frames.append(pd.DataFrame(benchmark_default_summary_rows))
if benchmark_calibrated_summary_rows:
    benchmark_summary_frames.append(pd.DataFrame(benchmark_calibrated_summary_rows))

benchmark_summary_export = pd.concat(benchmark_summary_frames, ignore_index=True) if benchmark_summary_frames else pd.DataFrame()
benchmark_predictions_export = (
    pd.concat(benchmark_calibrated_outputs, ignore_index=True) if benchmark_calibrated_outputs else pd.DataFrame()
)
benchmark_report_export = (
    pd.concat(benchmark_calibrated_reports, ignore_index=True) if benchmark_calibrated_reports else pd.DataFrame()
)

existing_benchmark_predictions = _existing_predictions_for_benchmark()
all_model_benchmark_predictions = pd.concat(
    [frame for frame in [existing_benchmark_predictions, benchmark_predictions_export] if not frame.empty],
    ignore_index=True,
) if (not existing_benchmark_predictions.empty or not benchmark_predictions_export.empty) else pd.DataFrame()
all_model_benchmark_summary = pd.DataFrame(
    _summaries_for_splits(all_model_benchmark_predictions, mode="all_model_benchmark_calibrated")
)

if benchmark_calibration_rows:
    display(pd.DataFrame(benchmark_calibration_rows))

if not all_model_benchmark_summary.empty:
    all_model_benchmark_summary = all_model_benchmark_summary.sort_values(
        ["eval_split", "macro_f1", "exact_duplicate_precision", "false_merge_rate"],
        ascending=[True, False, False, True],
    ).reset_index(drop=True)
    display(all_model_benchmark_summary)
    if not benchmark_report_export.empty:
        display(benchmark_report_export)

    all_model_benchmark_summary.to_csv(ALL_MODEL_BENCHMARK_SUMMARY_PATH, index=False)
    all_model_benchmark_predictions.to_csv(ALL_MODEL_BENCHMARK_PREDICTIONS_PATH, index=False)
    print(f"Saved all-model benchmark summary: {ALL_MODEL_BENCHMARK_SUMMARY_PATH} ({len(all_model_benchmark_summary)} rows)")
    print(f"Saved all-model benchmark predictions: {ALL_MODEL_BENCHMARK_PREDICTIONS_PATH} ({len(all_model_benchmark_predictions)} rows)")

    if not benchmark_summary_export.empty:
        benchmark_summary_export.to_csv(RERANKER_BENCHMARK_SUMMARY_PATH, index=False)
        benchmark_predictions_export.to_csv(RERANKER_BENCHMARK_PREDICTIONS_PATH, index=False)
        print(f"Saved reranker-only summary: {RERANKER_BENCHMARK_SUMMARY_PATH} ({len(benchmark_summary_export)} rows)")
        print(f"Saved reranker-only predictions: {RERANKER_BENCHMARK_PREDICTIONS_PATH} ({len(benchmark_predictions_export)} rows)")
else:
    print("All-model benchmark metrics skipped: no ready method predictions.")

## Как читать общий бенчмарк

Для выбора модели смотрим прежде всего строки `all_model_benchmark_calibrated / test` в таблице выше или в файле `all_model_benchmark_summary_sauces.csv`.

- `macro_f1` показывает общий баланс по двум классам.
- `exact_duplicate_precision` показывает, насколько аккуратно модель делает самый опасный auto-merge.
- `false_merge_rate` показывает долю случаев, где разные товары были ошибочно склеены как `exact_duplicate`.
- `pairs` показывает, на скольких парах считалась метрика. Если `MY_RERANKER_MAX_PAIRS = 120`, это быстрый пробный срез. Для финального выбора поставьте `0`.
- `seconds_per_pair` из предыдущей таблицы нужен для продуктового решения: модель может быть качественной, но слишком медленной для полного candidate set.

Если Qwen/BGE/Jina дают прирост по качеству, следующий шаг — не сразу тащить модель в production, а добавить supervised fusion: score reranker-модели + бренд + объём + pack + токены назначения/вкуса.